# Brainrot-V — evaluation

Scores the generated videos and produces every table and figure the thesis needs.
Run after `colab_runner.ipynb` has produced videos.

**Use a T4 runtime, not an A100.** Nothing here needs an A100 and a T4 costs
about an eighth as much per hour.

### Cells

| # | Does | Re-runnable |
| --- | --- | --- |
| 2 | Install VBench and weights | cached |
| 3 | Find videos, pair them by seed | instant |
| 4 | Local harness: flicker, speckle, streak, clean horizon | cached |
| 5 | VBench: six dimensions, **one at a time** | cached per dimension |
| 6 | DINO decomposition: long-range vs short-range drift | cached |
| 7 | Assemble one table of everything | instant |
| 8 | Statistics: sign test, Wilcoxon, Spearman, effect size | instant |
| 9 | Figures | instant |
| 10 | Markdown tables, ready to paste | instant |

Every expensive step writes its result to Drive and skips on the next run, so
re-running the notebook costs nothing and a dead session loses nothing.

### No single point of failure

Dimensions are scored **separately**, each in its own try/except. `aesthetic_quality`
in particular needs a weights file VBench does not download automatically — if it
fails, the other five still complete and the analysis proceeds without it. A
missing dimension shows as `—` in the tables rather than stopping the notebook.

## 1 · Settings

In [ ]:
ROOT    = "/content/drive/MyDrive/brainrot-v"
RUN_SET = "pilot"          # which videos to analyse: "bridge" | "pilot" | "final" | "all"

# VBench reloads its model per call, so videos are scored in batches. Smaller
# batches lose less work when a session dies; larger ones reload less often.
VBENCH_BATCH = 4

# VBench dimensions. Order matters only for display. Any that fail are skipped.
DIMENSIONS = [
    "subject_consistency",      # DINO   - identity drift        PRIMARY
    "background_consistency",   # CLIP   - scene stability       PRIMARY
    "motion_smoothness",        # AMT    - temporal artefacts
    "dynamic_degree",           # RAFT   - motion present        CONTROL
    "imaging_quality",          # MUSIQ  - per-frame quality
    "aesthetic_quality",        # LAION  - often needs a manual weights file
]

FLICKER_WINDOW = (1, 24)   # seconds averaged for the headline flicker number
DINO_STRIDE    = 2         # frame stride for the decomposition; 1 is exact, 2 is 2x faster

## 2 · Setup

In [ ]:
import json, os, re, shutil, subprocess, sys, warnings
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

REPO = Path("/content/causal_forcing")
if not REPO.exists():                          # same zip the runner notebook uses
    import zipfile
    with zipfile.ZipFile(Path(ROOT) / "00_setup" / "code" / "causal_forcing.zip") as zf:
        zf.extractall("/content")
os.chdir(REPO)

sys.path.insert(0, str(REPO / "notebooks"))
import colab_lib as cl
ws = cl.Workspace(ROOT).ensure()
print("workspace:", ws.root)

print("installing vbench (a few minutes, cached by Colab within a session) ...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vbench"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "matplotlib", "pandas", "scipy"], check=True)

# VBench's constructor demands a path to VBench_full_info.json. In custom_input
# mode the file is never read, but the argument is positional with no default.
FULL_INFO = Path("/content/VBench_full_info.json")
if not FULL_INFO.exists():
    subprocess.run(["wget", "-q", "-O", str(FULL_INFO),
                    "https://raw.githubusercontent.com/Vchitect/VBench/master/"
                    "vbench/VBench_full_info.json"], check=False)
    if not FULL_INFO.exists() or FULL_INFO.stat().st_size == 0:
        FULL_INFO.write_text("[]")             # a stub is enough for custom_input

# Weights land here and are large; keeping them on Drive avoids re-downloading
# them in every session.
os.environ["VBENCH_CACHE_DIR"] = str(ws.cache / "vbench_weights")
(ws.cache / "vbench_weights").mkdir(parents=True, exist_ok=True)

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE,
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 3 · Find and pair the videos

Reads whatever is in `outputs/`, plus anything you put in `reference/` — that is
where the eight existing `m0` clips from Modal go if you want to pair against
them rather than regenerate.

Pairs are formed on **(prompt, seed, duration)**, so each comparison holds
everything constant except the arm.

In [ ]:
import pandas as pd

SETS = cl.RUN_SETS if RUN_SET == "all" else (RUN_SET,)
SOURCES = [(ws.runs / s / "video", s) for s in SETS]
# Pairing a pilot needs its partner arm, which lives in 02_reference/ (clips
# generated elsewhere) or came from the bridge set.
SOURCES += [(ws.reference, "reference"), (ws.runs / "bridge" / "video", "bridge")]

# run_id -> (arm, prompt key, seed, seconds). Covers the runner's three sets and
# the original Modal names, so Modal clips can be dropped into reference/ as-is.
PATTERNS = [
    (r"^bridge-m0-s(\d+)-(\d+)s$",               lambda m: ("brainrotv", "p0", int(m[1]), int(m[2]))),
    (r"^pilot-baseline-s(\d+)-(\d+)s$",          lambda m: ("baseline",  "p0", int(m[1]), int(m[2]))),
    (r"^final-(\w+)-p(\d+)-s(\d+)-(\d+)s$",      lambda m: (m[1], f"p{m[2]}", int(m[3]), int(m[4]))),
    (r"^llm-m0-baseline-s(\d+)-(\d+)s$",         lambda m: ("brainrotv", "p0", int(m[1]), int(m[2]))),
]


def classify(stem):
    for pattern, build in PATTERNS:
        match = re.match(pattern, stem)
        if match:
            return build(match)
    return None


records, seen = [], set()
for folder, origin in SOURCES:
    if not folder.exists():
        continue
    for video in sorted(folder.glob("*.mp4")):
        if video.stem in seen:
            continue
        parsed = classify(video.stem)
        if not parsed:
            print(f"  skipped (unrecognised name): {video.name}")
            continue
        seen.add(video.stem)
        arm, prompt, seed, seconds = parsed
        meta = {}
        for candidate in cl.RUN_SETS:
            meta = cl.read_json(ws.manifest(candidate, video.stem), {}) or meta
            if meta:
                break
        records.append(dict(
            run_id=video.stem, path=str(video), origin=origin, arm=arm,
            prompt=prompt, seed=seed, seconds=seconds,
            memory_path=meta.get("memory_path"), flash_attn=meta.get("flash_attn"),
            wall_seconds=meta.get("wall_seconds"), peak_vram_gb=meta.get("peak_vram_gb"),
        ))

videos = pd.DataFrame(records)
if videos.empty:
    raise SystemExit("No videos found. Run colab_runner.ipynb first.")

# Pair: one row per (prompt, seed, seconds) that has both arms.
wide = videos.pivot_table(index=["prompt", "seed", "seconds"], columns="arm",
                          values="run_id", aggfunc="first")
PAIRS = wide.dropna(subset=[c for c in ("baseline", "brainrotv") if c in wide]).reset_index() \
    if {"baseline", "brainrotv"} <= set(wide.columns) else pd.DataFrame()

print(f"{len(videos)} videos, {len(PAIRS)} complete pairs\n")
print(videos[["run_id", "origin", "arm", "prompt", "seed", "memory_path",
              "flash_attn"]].to_string(index=False))

paths = set(videos.memory_path.dropna()); kernels = set(videos.flash_attn.dropna())
if len(paths) > 1 or len(kernels) > 1:
    warnings.warn(f"Mixed execution paths {paths} / {kernels}. Runs produced on "
                  "different hardware or attention kernels are not comparable.")

if PAIRS.empty:
    print("\nNo complete pairs yet - scoring still works, comparisons will be skipped.")
else:
    print(f"\npairs:\n{PAIRS.to_string(index=False)}")

## 4 · Local harness

`video_metrics.py`, unchanged from stages 1–17, so these numbers stay comparable
with everything already in the logbook.

In [ ]:
harness = cl.Checkpoint(ws.score(f"harness-{RUN_SET}"))
todo = [r for r in videos.itertuples() if not harness.done(r.run_id)]

# Batched, so an interrupted session keeps the batches it finished.
for start in range(0, len(todo), VBENCH_BATCH):
    batch = todo[start:start + VBENCH_BATCH]
    print(f"  measuring {start + 1}-{start + len(batch)} of {len(todo)} ...", flush=True)
    tmp = Path("/content/harness.json")
    subprocess.run([sys.executable, "video_metrics.py", str(tmp),
                    *[r.path for r in batch]], cwd=REPO, check=True)
    by_name = {Path(e["file"]).stem: e for e in json.loads(tmp.read_text())}
    harness.update({r.run_id: by_name[Path(r.path).stem]
                    for r in batch if Path(r.path).stem in by_name})

scored = harness.data
print(f"harness: {len(scored)} video(s) scored  ->  {harness.path.name}")


def harness_row(run_id):
    entry = scored.get(run_id)
    if not entry:
        return {}
    lo, hi = FLICKER_WINDOW
    out = {"clean_horizon": entry.get("clean_horizon_seconds")}
    for metric in ("flicker", "speckle", "streak"):
        series = entry["metrics"][metric]["per_second"][lo:hi]
        out[metric] = sum(series) / len(series) if series else None
    return out

## 5 · VBench

Each dimension is scored on its own and cached. A dimension that fails — missing
weights, an unsupported video — is reported and skipped; the rest still run.

In [ ]:
from vbench import VBench

# Checkpointed per (dimension, video), so a session that dies part-way through a
# dimension keeps the videos it already scored rather than repeating them.
scores = cl.Checkpoint(ws.score(f"vbench-{RUN_SET}"))
key = lambda dimension, run_id: f"{dimension}|{run_id}"
failed = {}

for dimension in DIMENSIONS:
    pending = [r for r in videos.itertuples()
               if not scores.done(key(dimension, r.run_id))]
    have = len(videos) - len(pending)
    if not pending:
        print(f"{dimension:<24} cached ({have} videos)")
        continue

    print(f"{dimension:<24} {have} cached, {len(pending)} to score", flush=True)
    for start in range(0, len(pending), VBENCH_BATCH):
        batch = pending[start:start + VBENCH_BATCH]
        stage = Path(f"/content/vb_stage/{dimension}")
        if stage.exists():
            shutil.rmtree(stage)
        stage.mkdir(parents=True)
        for row in batch:
            shutil.copy2(row.path, stage / f"{row.run_id}.mp4")

        # Folder mode asserts every video has a prompt. None of the six
        # custom-input dimensions read it - they are all reference-free - but
        # the entry must exist.
        prompts = {f"{r.run_id}.mp4": "a fox in a forest" for r in batch}
        out = Path(f"/content/vb_out/{dimension}")
        try:
            VBench(DEVICE, str(FULL_INFO), str(out)).evaluate(
                videos_path=str(stage), name=dimension, prompt_list=prompts,
                dimension_list=[dimension], mode="custom_input")
            payload = json.loads((out / f"{dimension}_eval_results.json").read_text())
            # {dimension: [aggregate, [{video_path, video_results}, ...]]}
            _, per_video = payload[dimension]
            scores.update({key(dimension, Path(v["video_path"]).stem): v["video_results"]
                           for v in per_video})
            print(f"{' ':<24} batch {start // VBENCH_BATCH + 1}: "
                  f"{len(per_video)} scored", flush=True)
        except Exception as error:
            failed[dimension] = f"{type(error).__name__}: {error}"
            print(f"{dimension:<24} FAILED - {failed[dimension]}")
            print(f"{' ':<24} skipping; it will show as '-' in the tables")
            break

# Regroup the flat checkpoint into {dimension: {run_id: score}}.
vbench = {}
for flat, value in scores.data.items():
    dimension, run_id = flat.split("|", 1)
    vbench.setdefault(dimension, {})[run_id] = value

print(f"\n{len(vbench)} of {len(DIMENSIONS)} dimensions have scores "
      f"({len(scores)} video-dimension pairs cached)")
for dimension, reason in failed.items():
    print(f"  FAILED  {dimension}: {reason}")

## 6 · Long-range vs short-range drift

VBench's `subject_consistency` is, per frame,
`(sim_to_previous + sim_to_first) / 2`.

Only the **`sim_to_first`** half measures drift away from the opening frame —
which is the Brainrot-V claim. The other half is short-range, and both arms keep
recent frames, so both score well on it. **The official metric dilutes the effect
by half**, and this cell recovers the two components.

Independent reimplementation using the same DINO ViT-B/16 backbone. Report it as
a decomposition, never as "VBench subject consistency".

In [ ]:
import numpy as np

store = cl.Checkpoint(ws.score(f"dino-{RUN_SET}"))
dino = store.data
pending = [r for r in videos.itertuples() if not store.done(r.run_id)]

if pending:
    import cv2
    import torch.nn.functional as F
    from torchvision import transforms

    model = torch.hub.load("facebookresearch/dino:main", "dino_vitb16").to(DEVICE).eval()
    prep = transforms.Compose([
        transforms.ToTensor(),
        transforms.Resize(224, antialias=True),
        transforms.CenterCrop(224),
        transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
    ])

    @torch.no_grad()
    def decompose(path):
        capture = cv2.VideoCapture(path)
        first = previous = None
        to_first, to_prev = [], []
        index = 0
        while True:
            ok, frame = capture.read()
            if not ok:
                break
            if index % DINO_STRIDE:
                index += 1
                continue
            index += 1
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            feature = F.normalize(model(prep(rgb).unsqueeze(0).to(DEVICE)), dim=-1)
            if first is None:
                first = feature
            else:
                to_first.append(max(0.0, F.cosine_similarity(first, feature).item()))
                to_prev.append(max(0.0, F.cosine_similarity(previous, feature).item()))
            previous = feature
        capture.release()
        return {"sim_to_first": float(np.mean(to_first)),
                "sim_to_prev": float(np.mean(to_prev)),
                "composite": float(np.mean(to_first) + np.mean(to_prev)) / 2,
                "to_first_series": [round(x, 5) for x in to_first]}

    for n, row in enumerate(pending, 1):
        print(f"  [{n}/{len(pending)}] {row.run_id}", flush=True)
        store.put(row.run_id, decompose(row.path))   # saved after each video

print(f"decomposition: {len(dino)} video(s)")
for run_id, value in sorted(dino.items()):
    print(f"  {run_id:<34} first {value['sim_to_first']:.4f}   "
          f"prev {value['sim_to_prev']:.4f}   composite {value['composite']:.4f}")

## 7 · One table of everything

In [ ]:
rows = []
for row in videos.itertuples():
    entry = dict(run_id=row.run_id, arm=row.arm, prompt=row.prompt, seed=row.seed,
                 seconds=row.seconds, origin=row.origin,
                 wall_seconds=row.wall_seconds, peak_vram_gb=row.peak_vram_gb)
    entry.update(harness_row(row.run_id))
    for dimension in DIMENSIONS:
        entry[dimension] = vbench.get(dimension, {}).get(row.run_id)
    decomposed = dino.get(row.run_id, {})
    entry["sim_to_first"] = decomposed.get("sim_to_first")
    entry["sim_to_prev"] = decomposed.get("sim_to_prev")
    rows.append(entry)

DATA = pd.DataFrame(rows).sort_values(["prompt", "seed", "arm"]).reset_index(drop=True)
DATA.to_csv(ws.table(f"per-video-{RUN_SET}.csv"), index=False)

METRICS = [d for d in DIMENSIONS if d in vbench] + \
          ["sim_to_first", "sim_to_prev", "flicker", "speckle", "streak", "clean_horizon"]
HIGHER_IS_BETTER = {*DIMENSIONS, "sim_to_first", "sim_to_prev", "clean_horizon"}

pd.set_option("display.width", 200, "display.max_columns", 40)
print(DATA[["run_id", "arm", "seed", *[m for m in METRICS if m in DATA]]]
      .to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print(f"\nwrote {ws.table(f'per-video-{RUN_SET}.csv')}")

## 8 · Statistics

Paired within seed, because seed difficulty dominates everything else in this
project — the measured spread is 3.0–9.2×.

The pre-registered primary test is a two-sided sign test on
`subject_consistency`, and the pre-registered threshold is **13 of 16**
(p = 0.021). Effect size is reported with every p-value; a win count alone is
not a result.

In [ ]:
from math import comb
from scipy import stats


def sign_test(wins, n):
    if n == 0:
        return float("nan")
    lo = min(wins, n - wins)
    return min(1.0, 2 * sum(comb(n, i) for i in range(lo + 1)) / 2 ** n)


def paired(metric):
    """One row per pair: baseline value, brainrot-v value, delta, win flag."""
    out = []
    for pair in PAIRS.itertuples():
        base = DATA[DATA.run_id.eq(pair.baseline)][metric]
        prop = DATA[DATA.run_id.eq(pair.brainrotv)][metric]
        if base.empty or prop.empty or pd.isna(base.iloc[0]) or pd.isna(prop.iloc[0]):
            continue
        b, p = float(base.iloc[0]), float(prop.iloc[0])
        better = p > b if metric in HIGHER_IS_BETTER else p < b
        out.append(dict(prompt=pair.prompt, seed=pair.seed, baseline=b,
                        brainrotv=p, delta=p - b,
                        ratio=(p / b if b else float("nan")),
                        tie=(p == b), win=bool(better)))
    return pd.DataFrame(out)


def live_pairs(table):
    """Exact ties carry no sign, so a sign test must drop them rather than count
    them as losses - otherwise an all-tied metric reports a spurious p = 0.008."""
    return table[~table.tie]


SUMMARY, PAIRED = [], {}
if PAIRS.empty:
    print("No complete pairs - nothing to test yet.")
else:
    for metric in METRICS:
        table = paired(metric)
        if table.empty:
            continue
        PAIRED[metric] = table
        live = live_pairs(table)
        n, wins, ties = len(live), int(live.win.sum()), int(table.tie.sum())
        p_sign = sign_test(wins, n) if n else float("nan")
        try:
            p_wilcoxon = (stats.wilcoxon(live.baseline, live.brainrotv).pvalue
                          if n else float("nan"))
        except ValueError:
            p_wilcoxon = float("nan")
        SUMMARY.append(dict(
            metric=metric, n=n, ties=ties, wins=wins,
            win_rate=(wins / n if n else float("nan")),
            baseline=table.baseline.mean(), brainrotv=table.brainrotv.mean(),
            delta=table.delta.mean(), median_delta=table.delta.median(),
            p_sign=p_sign, p_wilcoxon=p_wilcoxon))

    SUMMARY = pd.DataFrame(SUMMARY)
    print(SUMMARY.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

    primary = "subject_consistency"
    if primary in PAIRED:
        table = live_pairs(PAIRED[primary])
        n, wins = len(table), int(table.win.sum())
        need = next((k for k in range(n, n // 2, -1) if sign_test(k, n) < 0.05), None)
        print(f"\nPRIMARY - {primary}")
        print(f"  {wins} of {n} paired wins, two-sided sign test p = {sign_test(wins, n):.4f}")
        print(f"  minimum for p < 0.05 at n = {n} is {need}/{n}"
              if need else "  no win count reaches p < 0.05 at this n")
        print(f"  mean delta {table.delta.mean():+.4f}   "
              f"median {table.delta.median():+.4f}   "
              f"range {table.delta.min():+.4f} to {table.delta.max():+.4f}")

        # Does the effect depend on how hard the seed was? This is the structure
        # that was significant (rho = -0.81) on the eviction data while the
        # average effect was not.
        if len(table) >= 4:
            rho, p_rho = stats.spearmanr(table.baseline, table.delta)
            print(f"  Spearman rho(baseline, delta) = {rho:+.3f}, p = {p_rho:.4f}"
                  + ("   <- conditional effect" if p_rho < 0.05 else ""))

    control = "dynamic_degree"
    if control in PAIRED:
        table = live_pairs(PAIRED[control])
        n = len(table)
        reduced = int((~table.win).sum())            # pairs with LESS motion
        p_ctrl = sign_test(reduced, n) if n else float("nan")
        print(f"\nCONTROL - {control}")
        print(f"  motion reduced in {reduced} of {n} pairs, "
              f"mean delta {table.delta.mean():+.4f}, sign test p = {p_ctrl:.4f}")
        if reduced > n / 2 and p_ctrl < 0.05:
            print("  *** Motion is significantly REDUCED. A consistency win alongside")
            print("      this means the method stabilises by freezing - report it as")
            print("      a trade, not a win.")
        else:
            print("  OK - no significant loss of motion, so any consistency gain is")
            print("     not explained by the video simply moving less.")

## 9 · Figures

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

BASE_C, PROP_C, GRID = "#b4453c", "#2f6f9f", "#d8d8d8"
FIGS = ws.figures


def tidy(ax):
    ax.grid(axis="y", color=GRID, linewidth=0.6)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)


def save(fig, name):
    path = FIGS / f"{name}-{RUN_SET}.png"
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print("  ", path.name)
    return path


print("figures written to", FIGS)

# --- 1. paired lines, primary metric ---------------------------------------
if "subject_consistency" in PAIRED:
    table = PAIRED["subject_consistency"].sort_values(["prompt", "seed"])
    labels = [f"{r.prompt}/s{r.seed}" for r in table.itertuples()]
    x = range(len(table))
    fig, ax = plt.subplots(figsize=(max(7, len(table) * 0.8), 4.2))
    ax.plot(x, table.baseline, "o-", color=BASE_C, label="baseline (rolling)", lw=2)
    ax.plot(x, table.brainrotv, "s-", color=PROP_C, label="Brainrot-V", lw=2)
    for i, row in enumerate(table.itertuples()):
        ax.plot([i, i], [row.baseline, row.brainrotv], color="#999", lw=1, zorder=0)
    wins, n = int(table.win.sum()), len(table)
    ax.set_xticks(list(x)); ax.set_xticklabels(labels, fontsize=8, rotation=45, ha="right")
    ax.set_ylabel("subject consistency (higher better)", fontsize=9)
    ax.set_title(f"Paired by seed: {wins} of {n} wins, p = {sign_test(wins, n):.3f}",
                 fontsize=10)
    ax.legend(fontsize=9, frameon=False); tidy(ax)
    save(fig, "paired-subject-consistency")

# --- 2. effect against difficulty ------------------------------------------
if "subject_consistency" in PAIRED and len(PAIRED["subject_consistency"]) >= 4:
    table = PAIRED["subject_consistency"]
    fig, ax = plt.subplots(figsize=(5.5, 4.2))
    ax.scatter(table.baseline, table.delta,
               c=[PROP_C if w else BASE_C for w in table.win], s=70, zorder=3)
    for row in table.itertuples():
        ax.annotate(f"s{row.seed}", (row.baseline, row.delta),
                    textcoords="offset points", xytext=(7, 3), fontsize=8)
    ax.axhline(0, color="#888", lw=1, ls="--")
    rho, p_rho = stats.spearmanr(table.baseline, table.delta)
    ax.set_xlabel("baseline score (lower = harder seed)", fontsize=9)
    ax.set_ylabel("delta (Brainrot-V - baseline)", fontsize=9)
    ax.set_title(f"Effect vs difficulty: rho = {rho:+.2f}, p = {p_rho:.3f}", fontsize=10)
    tidy(ax)
    save(fig, "effect-vs-difficulty")

# --- 3. every dimension, mean with per-pair points -------------------------
if SUMMARY is not None and len(SUMMARY):
    dims = [m for m in SUMMARY.metric if m in DIMENSIONS]
    if dims:
        fig, axes = plt.subplots(1, len(dims), figsize=(2.3 * len(dims), 4.2),
                                 squeeze=False)
        for ax, metric in zip(axes[0], dims):
            table = PAIRED[metric]
            # A bar from zero would hide the effect: these scores all sit near
            # 0.9 and the difference lives in the third decimal. One line per
            # pair, on a y-range fitted to the data.
            for row in table.itertuples():
                ax.plot([0, 1], [row.baseline, row.brainrotv],
                        color=(PROP_C if row.win else BASE_C),
                        lw=1.1, marker="o", ms=4, alpha=0.65, zorder=2)
            ax.plot([0, 1], [table.baseline.mean(), table.brainrotv.mean()],
                    color="#222", lw=2.4, marker="D", ms=6, zorder=4, label="mean")
            values = list(table.baseline) + list(table.brainrotv)
            pad = (max(values) - min(values) or 0.01) * 0.12
            ax.set_ylim(min(values) - pad, max(values) + pad)
            live = live_pairs(table)
            wins, n = int(live.win.sum()), len(live)
            ax.set_xlim(-0.3, 1.3)
            ax.set_xticks([0, 1]); ax.set_xticklabels(["base", "B-V"], fontsize=9)
            ax.tick_params(axis="y", labelsize=7)
            ax.set_title(f"{metric.replace('_', chr(10))}\n{wins}/{n}  "
                         f"p={sign_test(wins, n):.3f}" if n else metric, fontsize=8)
            tidy(ax)
        axes[0][0].legend(fontsize=8, frameon=False)
        save(fig, "all-dimensions")

# --- 4. the decomposition ---------------------------------------------------
if "sim_to_first" in PAIRED and "sim_to_prev" in PAIRED:
    fig, ax = plt.subplots(figsize=(6, 4.2))
    parts = ["sim_to_first", "sim_to_prev"]
    offsets = [-0.17, 0.17]
    for shift, arm, colour in ((o, a, c) for o, (a, c) in
                               zip(offsets, (("baseline", BASE_C), ("brainrotv", PROP_C)))):
        ax.bar([i + shift for i in range(len(parts))],
               [PAIRED[p][arm].mean() for p in parts], width=0.33,
               color=colour, label=arm, zorder=2)
    for i, part in enumerate(parts):                  # put the gap on the chart
        t = PAIRED[part]
        top = max(t.baseline.mean(), t.brainrotv.mean())
        ax.annotate(f"{t.delta.mean():+.4f}", (i, top), ha="center",
                    textcoords="offset points", xytext=(0, 6), fontsize=10,
                    fontweight="bold",
                    color=PROP_C if t.delta.mean() > 0 else BASE_C)
    ax.set_ylim(0, max(PAIRED[p].brainrotv.mean() for p in parts) * 1.18)
    ax.set_xticks(range(len(parts)))
    ax.set_xticklabels(["sim to FIRST frame\n(long-range - the claim)",
                        "sim to PREVIOUS frame\n(short-range)"], fontsize=9)
    ax.set_ylabel("DINO cosine similarity", fontsize=9)
    ax.set_title("The official metric averages these two", fontsize=10)
    ax.legend(fontsize=9, frameon=False, loc="lower right"); tidy(ax)
    save(fig, "decomposition")

# --- 5. drift over time -----------------------------------------------------
series = {r.run_id: dino[r.run_id]["to_first_series"]
          for r in videos.itertuples() if r.run_id in dino}
if series:
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    for row in videos.itertuples():
        values = series.get(row.run_id)
        if not values:
            continue
        seconds = np.arange(len(values)) * DINO_STRIDE / 16
        ax.plot(seconds, values, lw=1.2, alpha=0.75,
                color=PROP_C if row.arm == "brainrotv" else BASE_C)
    ax.plot([], [], color=BASE_C, label="baseline")
    ax.plot([], [], color=PROP_C, label="Brainrot-V")
    ax.set_xlabel("seconds", fontsize=9)
    ax.set_ylabel("similarity to first frame", fontsize=9)
    ax.set_title("Identity drift over the rollout", fontsize=10)
    ax.legend(fontsize=9, frameon=False); tidy(ax)
    save(fig, "drift-over-time")

# --- 6. flicker over time ---------------------------------------------------
if scored:
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    for row in videos.itertuples():
        entry = scored.get(row.run_id)
        if not entry:
            continue
        values = entry["metrics"]["flicker"]["per_second"]
        ax.plot(range(len(values)), values, lw=1.2, alpha=0.75,
                color=PROP_C if row.arm == "brainrotv" else BASE_C)
    ax.plot([], [], color=BASE_C, label="baseline")
    ax.plot([], [], color=PROP_C, label="Brainrot-V")
    ax.set_xlabel("seconds", fontsize=9)
    ax.set_ylabel("flicker (lower better)", fontsize=9)
    ax.set_title("Flicker across the rollout", fontsize=10)
    ax.legend(fontsize=9, frameon=False); tidy(ax)
    save(fig, "flicker-over-time")

## 10 · Tables

Writes GitHub-flavoured Markdown straight into `tables/`, ready to paste into
abdanslog or the thesis. Blank cells mean a dimension did not score, not a zero.

In [ ]:
def fmt(value, places=4):
    return "—" if value is None or (isinstance(value, float) and pd.isna(value)) \
        else f"{value:.{places}f}"


lines = [f"# Brainrot-V evaluation — `{RUN_SET}`", ""]
lines += [f"{len(videos)} videos, {len(PAIRS)} paired comparisons.",
          f"Dimensions scored: {', '.join(vbench) or 'none'}.", ""]

# Table 1 - headline
lines += ["## Table 1 — Headline", "",
          "| Metric | Baseline | Brainrot-V | Δ | Paired wins | Sign test p |",
          "| --- | ---: | ---: | ---: | :---: | ---: |"]
for row in (SUMMARY.itertuples() if len(SUMMARY) else []):
    flag = " **(control)**" if row.metric == "dynamic_degree" else ""
    ties = f" (+{row.ties} tied)" if row.ties else ""
    lines.append(f"| `{row.metric}`{flag} | {fmt(row.baseline)} | {fmt(row.brainrotv)} "
                 f"| {row.delta:+.4f} | {row.wins}/{row.n}{ties} | {fmt(row.p_sign, 4)} |")

# Table 2 - decomposition
lines += ["", "## Table 2 — subject_consistency decomposed", "",
          "| Component | Baseline | Brainrot-V | Δ | Note |",
          "| --- | ---: | ---: | ---: | --- |"]
for part, note in (("sim_to_first", "long-range anchor — the claim"),
                   ("sim_to_prev", "short-range; both arms keep recent frames")):
    if part in PAIRED:
        t = PAIRED[part]
        lines.append(f"| `{part}` | {fmt(t.baseline.mean())} | {fmt(t.brainrotv.mean())} "
                     f"| {t.delta.mean():+.4f} | {note} |")

# Table 3 - per pair
lines += ["", "## Table 3 — Every pair", "",
          "| Prompt | Seed | Baseline | Brainrot-V | Δ | Win |",
          "| --- | ---: | ---: | ---: | ---: | :---: |"]
for row in (PAIRED.get("subject_consistency", pd.DataFrame()).itertuples()):
    lines.append(f"| {row.prompt} | {row.seed} | {fmt(row.baseline)} | "
                 f"{fmt(row.brainrotv)} | {row.delta:+.4f} | "
                 f"{'yes' if row.win else '**no**'} |")

# Table 4 - local harness
lines += ["", "## Table 4 — Local harness", "",
          "| Metric | Baseline | Brainrot-V | Δ | Paired wins |",
          "| --- | ---: | ---: | ---: | :---: |"]
for metric in ("flicker", "speckle", "streak", "clean_horizon"):
    if metric in PAIRED:
        t = PAIRED[metric]
        lines.append(f"| {metric} | {fmt(t.baseline.mean())} | {fmt(t.brainrotv.mean())} "
                     f"| {t.delta.mean():+.4f} | {int(t.win.sum())}/{len(t)} |")

# Table 5 - efficiency and provenance
lines += ["", "## Table 5 — Efficiency and provenance", "",
          "| Arm | Videos | Mean wall (s) | Peak VRAM (GB) | Memory path | Attention |",
          "| --- | ---: | ---: | ---: | --- | --- |"]
for arm, group in videos.groupby("arm"):
    lines.append(f"| {arm} | {len(group)} | {fmt(group.wall_seconds.mean(), 0)} | "
                 f"{fmt(group.peak_vram_gb.mean(), 1)} | "
                 f"{'/'.join(sorted(set(group.memory_path.dropna()))) or '—'} | "
                 f"{'/'.join(sorted(set(group.flash_attn.dropna()))) or 'sdpa'} |")

lines += ["", "## Active KV tokens", "",
          "| | Baseline | Brainrot-V | Δ |", "| --- | ---: | ---: | ---: |",
          "| Active KV tokens (21-frame eval) | 32,760 | 22,328 | **−31.8%** |"]

report = ws.table(f"report-{RUN_SET}.md")
cl.write_atomic(report, "\n".join(lines))
print("\n".join(lines))
ws.write_status()
print(f"\n\nwrote   {report}")
print(f"figures {FIGS}")
print(f"status  {ws.root / 'STATUS.md'}")